# Handwired three GPU research previews — native T4 shader gate
Strictly validates all five real WGSL render entrypoints for Cataclysm, Destruction and Machines on physical NVIDIA T4. Compilation is not visual QA; screenshots and browser tests remain separately required.


In [ ]:
import subprocess,pathlib,json,traceback,time
root=pathlib.Path('/kaggle/working')
job={'ok':False,'stages':[]}
def execute(title,command):
 print('STEP',title,flush=True)
 proc=subprocess.run(command,shell=True,executable='/bin/bash',text=True,
  stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=1000)
 print(proc.stdout[-4500:],flush=True)
 job['stages'].append({'title':title,'exitCode':proc.returncode,'tail':proc.stdout[-1500:]})
 (root/'frontiers-visual-job.json').write_text(json.dumps(job,indent=2))
 if proc.returncode:raise RuntimeError(title+' failed')
try:
 execute('Identify actual NVIDIA hardware','nvidia-smi --query-gpu=name,driver_version --format=csv,noheader')
 execute('Install NVIDIA Vulkan and Python backend','apt-get update -qq && apt-get install -y -qq libnvidia-gl-580 libvulkan1 vulkan-tools && python -m pip -q install wgpu==0.32.0')
 execute('Install Bun','curl -fsSL https://bun.sh/install | bash')
 execute('Fetch exact unified Frontiers source','git clone --depth 1 --branch integration/eight-frontiers-fast https://github.com/Chrisbryan17/handwired.git /kaggle/temp/handwired-allvisuals')
 execute('Install pinned compiler deps','cd /kaggle/temp/handwired-allvisuals && /root/.bun/bin/bun install --frozen-lockfile')
 execute('Export all three exact WGSL modules','cd /kaggle/temp/handwired-allvisuals && /root/.bun/bin/bun scripts/export-frontiers-visual.ts /kaggle/working')
 execute('NVIDIA T4 compilation + pipeline linking','python /kaggle/temp/handwired-allvisuals/ops/kaggle/frontiers-visual/verify_render.py /kaggle/working')
 job['evidence']=json.loads((root/'frontiers-native-visual-pipelines.json').read_text())
 job['ok']=job['evidence'].get('ok',False)
except Exception as error:
 job['error']=str(error);traceback.print_exc()
finally:
 job['finished']=time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())
 (root/'frontiers-visual-job.json').write_text(json.dumps(job,indent=2))
print('FRONTIERS_VISUAL_SHADER_GATE',job['ok'],flush=True)
